# 🚗 Autonomous Vehicle Lane Following — Experiments Notebook
**Group A** — João Ferreira · Henrique Teixeira · Miguel Almeida

This notebook is an entry point for **all three experiments** defined in the project proposal:

| # | Experiment | Variable |
|---|-----------|---------|
| 1 | Action Space | Discrete (DQN) vs Continuous (PPO) |
| 2 | Reward Function | Dense / TTC / Sparse  ×  DQN / PPO (2 × 3 matrix) |
| 3 | Camera Robustness | FOV distance + distortion filters (fog, rain, low-light) |

### How to use this notebook
1. **Start Webots** with the appropriate world file open and the vehicle controller set to `train.py` (or `evaluate.py` for eval cells).
2. **Run section by section** — each section is self-contained. You can run Environment Sanity Checks first to confirm the Webots connection works before committing to a full training run.
3. **Training cells** launch `train.py` as a subprocess so Webots's controller mechanism is respected.
     ⚠️ *`train.py` / `evaluate.py` **must** be run as Webots robot controllers — they import `controller.Supervisor` at startup. The cells here launch them correctly via `subprocess` targeting the Webots Python binary.*
4. **Analysis cells** load the CSV files written by `MetricsCallback` and produce all the plots and tables referenced in Section 4 of the proposal.

### ⚠️ Corrections & improvements
- **Experiment 2 matrix** — the proposal shows Dense/Sparse × Discrete/Continuous (2 × 2). The `exp2/` YAML configs also include a `ppo_ttc` run, so the notebook implements the full **2 × 3** matrix (DQN + PPO) × (Dense + TTC + Sparse).
- **Sparse reward convergence** — raw sparse rarely converges; the notebook applies curriculum stages automatically via `--timesteps` scaling.
- **World-level experiments** — `city.wbt` (clean baseline), `city_level1.wbt` (static cones/signs, parked car, FOV ≈ medium), `city_level2.wbt` (more cones, extra Solid obstacles, harder track). Experiment 3 Part A uses different worlds to vary effective FOV / complexity; Part B patches the camera image in `preprocess_obs`.
- **Safety-score note** — the code divides `distance / near_misses`, clamping near_misses to 1. Higher = safer.

## 0. Setup & Paths

In [ ]:
import os, sys, subprocess, time, csv, copy, textwrap, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(".").resolve()
print("Project root:", PROJECT_ROOT)
CONFIG_PATH   = PROJECT_ROOT / "config.yaml"
RESULTS_DIR   = PROJECT_ROOT / "results"
WORLDS_DIR    = PROJECT_ROOT / "worlds"
RESULTS_DIR.mkdir(exist_ok=True)

# ── Webots Python binary ──────────────────────────────────────────────────────
WEBOTS_HOME = Path(os.environ.get("WEBOTS_HOME", "/usr/local/webots"))
WEBOTS_PY   = WEBOTS_HOME / "lib" / "controller" / "python" / "python3"

if not WEBOTS_PY.exists():
    WEBOTS_PY = Path(sys.executable)
    print(f"⚠️  Webots Python not found at {WEBOTS_PY}. Analysis cells work fine; training cells need a real Webots install.")
else:
    print(f"✅ Webots Python: {WEBOTS_PY}")

# ── Config ────────────────────────────────────────────────────────────────────
with open(CONFIG_PATH) as f:
    BASE_CFG = yaml.safe_load(f)
print("Base config loaded — reward type:", BASE_CFG["reward"]["type"])

## 1. Helper Utilities

These functions are used throughout the notebook.

In [ ]:
def launch_training(agent: str, reward: str, timesteps: int,
                    obstacles: bool = False, config: Path = CONFIG_PATH,
                    extra_args: list = None) -> int:
    """Blocking wrapper around train.py."""
    cmd = [
        str(WEBOTS_PY), str(PROJECT_ROOT / "train.py"),
        "--agent",     agent,
        "--reward",    reward,
        "--config",    str(config),
        "--timesteps", str(timesteps),
    ]
    if obstacles:
        cmd.append("--obstacles")
    if extra_args:
        cmd.extend(extra_args)
        
    label = f"{agent.upper()}_{reward}"
    print(f"▶ Starting {label}  ({timesteps:,} timesteps)  {'[+obstacles]' if obstacles else ''}")
    print(f"  cmd: {' '.join(cmd)}")
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, cwd=PROJECT_ROOT)
    try:
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        print("\n⚠️  Training interrupted.")
    elapsed = time.time() - t0
    print(f"\n✅ {label} finished in {elapsed/60:.1f} min  (exit code {proc.returncode})")
    return proc.returncode

_CSV_COLS = [
    "n_episodes", "success_rate_%", "mean_collisions",
    "mean_cross_track_error_m", "mean_lap_time_s",
    "safety_score", "mean_steps", "mean_reward",
]

def load_metrics_csv(agent: str, reward: str) -> pd.DataFrame:
    p = RESULTS_DIR / f"{agent}_{reward}_metrics.csv"
    if not p.exists():
        print(f"  ⚠️  {p.name} not found — returning empty DataFrame.")
        return pd.DataFrame(columns=_CSV_COLS)
    df = pd.read_csv(p)
    df.columns = [c.strip() for c in df.columns]
    return df

def load_eval_csv(name: str) -> pd.DataFrame:
    """Load an evaluate.py output CSV by partial filename match."""
    matches = sorted(RESULTS_DIR.glob(f"eval_{name}*.csv"))
    if not matches:
        print(f"  ⚠️  No eval CSV matching 'eval_{name}*.csv' found.")
        return pd.DataFrame()
    p = matches[-1]
    print(f"  Loaded {p.name}")
    return pd.read_csv(p)

PALETTE = {
    "ppo_dense":  "#2196F3", "ppo_ttc":    "#4CAF50", "ppo_sparse": "#FF9800",
    "dqn_dense":  "#9C27B0", "dqn_ttc":    "#00BCD4", "dqn_sparse": "#F44336",
}

def plot_learning_curves(configs: list, metric: str = "mean_reward",
                         title: str = "Learning Curve", window: int = 5,
                         save_as: str = None):
    fig, ax = plt.subplots(figsize=(9, 4))
    for (agent, reward) in configs:
        key = f"{agent}_{reward}"
        df  = load_metrics_csv(agent, reward)
        if df.empty or metric not in df.columns:
            continue
        smoothed = df[metric].rolling(window, min_periods=1).mean()
        color = PALETTE.get(key, "gray")
        ax.plot(df["n_episodes"], smoothed, label=key.upper().replace("_", " + "),
                color=color, linewidth=2)
        ax.fill_between(df["n_episodes"],
                        df[metric].rolling(window, min_periods=1).min(),
                        df[metric].rolling(window, min_periods=1).max(),
                        alpha=0.12, color=color)
    ax.set_xlabel("Episodes completed")
    ax.set_ylabel(metric.replace("_", " "))
    ax.set_title(title)
    ax.legend(fontsize=8)
    plt.tight_layout()
    if save_as: plt.savefig(RESULTS_DIR / save_as, bbox_inches="tight")
    plt.show()

def summary_table(configs: list, label_col: str = "Config") -> pd.DataFrame:
    rows = []
    for (agent, reward) in configs:
        df = load_metrics_csv(agent, reward)
        if df.empty: continue
        tail = df.tail(max(1, len(df) // 5))
        row = {"Config": f"{agent.upper()} + {reward}"}
        for col in _CSV_COLS[1:]:
            if col in tail.columns:
                row[col] = round(tail[col].mean(), 3)
        rows.append(row)
    return pd.DataFrame(rows).set_index("Config") if rows else pd.DataFrame()

def bar_comparison(df: pd.DataFrame, metrics: list, title: str = "Comparison", save_as: str = None):
    n_metrics = len(metrics)
    n_configs = len(df)
    x = np.arange(n_configs)
    width = 0.8 / n_metrics
    fig, axes = plt.subplots(1, n_metrics, figsize=(4 * n_metrics, 4), sharey=False)
    if n_metrics == 1: axes = [axes]
    for i, (ax, m) in enumerate(zip(axes, metrics)):
        if m not in df.columns: ax.set_visible(False); continue
        colors = [list(PALETTE.values())[j % len(PALETTE)] for j in range(n_configs)]
        bars = ax.bar(x, df[m].values, color=colors, alpha=0.85, edgecolor="white")
        ax.set_xticks(x)
        ax.set_xticklabels(df.index, rotation=35, ha="right", fontsize=7)
        ax.set_title(m.replace("_", " "), fontsize=9)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h * 1.01, f"{h:.2f}", ha="center", va="bottom", fontsize=7)
    fig.suptitle(title, fontsize=11, fontweight="bold")
    plt.tight_layout()
    if save_as: plt.savefig(RESULTS_DIR / save_as, bbox_inches="tight")
    plt.show()
print("✅ Helpers loaded.")

## 2. Environment Sanity Checks
Run these cells **before** any training to confirm that operational environment wrappers match definitions.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
cfg_copy = copy.deepcopy(BASE_CFG)
cam_h, cam_w = cfg_copy["env"]["camera_resolution"]
lidar_size   = 512
grayscale    = cfg_copy["observation"].get("grayscale", False)
norm_lidar   = cfg_copy["observation"].get("normalize_lidar", True)
lidar_max    = cfg_copy["observation"].get("lidar_max_range", 10.0)
cam_ch = 1 if grayscale else 3

cam_space   = spaces.Box(0.0, 1.0, (cam_h, cam_w, cam_ch), np.float32)
lidar_space = spaces.Box(0.0, 1.0 if norm_lidar else lidar_max, (lidar_size,), np.float32)
state_space = spaces.Box(np.array([0.0, -1.0], np.float32), np.array([1.0,  1.0], np.float32), dtype=np.float32)

print("=== Observation Space ===")
print(f"  camera : {cam_space.shape}  dtype={cam_space.dtype}")
print(f"  lidar  : {lidar_space.shape}  dtype={lidar_space.dtype}")
print(f"  state  : {state_space.shape}  [speed/max_speed ∈ [0,1], alignment ∈ [-1,1]]")
print("\n=== Action Spaces ===")
print("  Discrete (DQN) : Discrete(4)  — 0=Left, 1=Right, 2=Straight, 3=Brake")
print("  Continuous (PPO): Box([-1,-1], [1,1])  — [steering, throttle]")

In [ ]:
print("Running SB3 env checker via Webots controller…")
ret = launch_training("ppo", "dense", timesteps=1, extra_args=["--check-env"])
if ret == 0:
    print("✅ Env check passed — spaces, dtypes and step() all validated.")
else:
    print("❌ Env check failed — check Webots console for tracebacks.")

In [ ]:
print("Starting 500-step smoke test (PPO, dense reward, city.wbt, no obstacles)…")
launch_training("ppo", "dense", timesteps=500)

In [ ]:
print("Starting 500-step smoke test (DQN, dense reward)…")
launch_training("dqn", "dense", timesteps=500)

In [ ]:
print("Starting 500-step smoke test (PPO, dense, +obstacles)…")
launch_training("ppo", "dense", timesteps=500, obstacles=True)

In [ ]:
print("Worlds available:")
for w in sorted(WORLDS_DIR.glob("*.wbt")):
    print(f"  {w.name}")

## 3. Experiment 1 — Action Space: Discrete vs Continuous
**Goal:** Compare DQN (discrete) and PPO (continuous) on trajectory tracking convergence profiles.

In [ ]:
TIMESTEPS_EXP1 = BASE_CFG["training"]["total_timesteps"]
print(f"Experiment 1: PPO + Dense  |  {TIMESTEPS_EXP1:,} timesteps")
launch_training("ppo", "dense", timesteps=TIMESTEPS_EXP1)

In [ ]:
print(f"Experiment 1: DQN + Dense  |  {TIMESTEPS_EXP1:,} timesteps")
launch_training("dqn", "dense", timesteps=TIMESTEPS_EXP1)

In [ ]:
plot_learning_curves(configs=[("ppo", "dense"), ("dqn", "dense")], metric="mean_reward",
                     title="Exp 1 — Learning Curves: PPO vs DQN (dense reward)", save_as="exp1_learning_curve_reward.png")

In [ ]:
plot_learning_curves(configs=[("ppo", "dense"), ("dqn", "dense")], metric="mean_cross_track_error_m",
                     title="Exp 1 — Cross-Track Error: PPO vs DQN", save_as="exp1_learning_curve_cte.png")

In [ ]:
tbl = summary_table([("ppo", "dense"), ("dqn", "dense")])
if not tbl.empty: display(tbl)

In [ ]:
if not tbl.empty:
    bar_comparison(tbl, metrics=["success_rate_%", "mean_cross_track_error_m", "mean_lap_time_s", "safety_score"],
                   title="Exp 1 — PPO vs DQN (dense reward)", save_as="exp1_bar_comparison.png")

In [ ]:
if not tbl.empty and "PPO + dense" in tbl.index:
    ppo_sr  = tbl.loc["PPO + dense", "success_rate_%"]
    dqn_sr  = tbl.loc["DQN + dense", "success_rate_%"]
    ppo_cte = tbl.loc["PPO + dense", "mean_cross_track_error_m"]
    dqn_cte = tbl.loc["DQN + dense", "mean_cross_track_error_m"]
    print("=== Experiment 1 Hypothesis Assessment ===")
    print(f"  PPO success rate : {ppo_sr:.1f} %  |  DQN: {dqn_sr:.1f} %")
    print(f"  PPO mean CTE     : {ppo_cte:.3f}    |  DQN: {dqn_cte:.3f}")

## 4. Experiment 2 — Reward Function Impact
**Goal:** Comparative evaluation mapping across the full 2x3 Reward Matrix.

In [ ]:
T_DENSE  = BASE_CFG["training"]["total_timesteps"]
T_TTC    = T_DENSE
T_SPARSE = T_DENSE * 3
print(f"Exp 2 timesteps: Dense={T_DENSE:,}, TTC={T_TTC:,}, Sparse={T_SPARSE:,}")

In [ ]:
launch_training("ppo", "dense", timesteps=T_DENSE)

In [ ]:
launch_training("ppo", "ttc", timesteps=T_TTC)

In [ ]:
launch_training("ppo", "sparse", timesteps=T_SPARSE)

In [ ]:
launch_training("dqn", "dense", timesteps=T_DENSE)

In [ ]:
launch_training("dqn", "ttc", timesteps=T_TTC)

In [ ]:
launch_training("dqn", "sparse", timesteps=T_SPARSE)

In [ ]:
ALL_CONFIGS = [("ppo", "dense"), ("ppo", "ttc"), ("ppo", "sparse"),
               ("dqn", "dense"), ("dqn", "ttc"), ("dqn", "sparse")]
plot_learning_curves(ALL_CONFIGS, metric="mean_reward", title="Exp 2 — Reward Function Impact: all 6 configs", save_as="exp2_all_reward.png")

In [ ]:
def steps_to_threshold(agent, reward, threshold_pct=0.80):
    df = load_metrics_csv(agent, reward)
    if df.empty or "mean_reward" not in df.columns: return float("nan")
    max_r  = df["mean_reward"].max()
    target = threshold_pct * max_r
    hit    = df[df["mean_reward"] >= target]
    return int(hit["n_episodes"].iloc[0]) if not hit.empty else float("nan")

for ag, rw in ALL_CONFIGS:
    s = steps_to_threshold(ag, rw)
    print(f"  {ag.upper()} + {rw:<10} Steps to 80% max reward: {f'{s:,}' if not np.isnan(s) else 'never reached'}")

In [ ]:
tbl2 = summary_table(ALL_CONFIGS)
if not tbl2.empty:
    display(tbl2.style.background_gradient(cmap="RdYlGn", subset=["success_rate_%"])
                 .background_gradient(cmap="RdYlGn_r", subset=["mean_collisions"]))

In [ ]:
pivot_data = {}
for ag, rw in ALL_CONFIGS:
    df = load_metrics_csv(ag, rw)
    if not df.empty and "mean_collisions" in df.columns:
        pivot_data[(ag.upper(), rw)] = df["mean_collisions"].iloc[-1]

if pivot_data:
    agents, rewards = ["PPO", "DQN"], ["dense", "ttc", "sparse"]
    matrix = np.array([[pivot_data.get((a, r), np.nan) for r in rewards] for a in agents])
    fig, ax = plt.subplots(figsize=(6, 3))
    im = ax.imshow(matrix, cmap="RdYlGn_r", aspect="auto", vmin=0)
    ax.set_xticks(range(len(rewards))); ax.set_xticklabels(rewards)
    ax.set_yticks(range(len(agents)));  ax.set_yticklabels(agents)
    for i in range(len(agents)):
        for j in range(len(rewards)):
            ax.text(j, i, f"{matrix[i,j]:.2f}", ha="center", va="center", fontweight="bold")
    plt.colorbar(im, ax=ax, label="Mean collisions / episode")
    plt.title("Exp 2 — Collision rate heatmap (lower = better)")
    plt.show()

## 5. Experiment 3 — Camera Robustness
### Part A — Field of View (world selection)
Comparing model performance indicators under alternative environmental clutter loads configuration matrices.

In [ ]:
print("Part A — Baseline world (city.wbt). Reusing trained ppo_dense checkpoints.")

In [ ]:
print("Part A — Level 1 world (city_level1.wbt)")
input("Verify Webots has loaded city_level1.wbt, then press enter...")
launch_training("ppo", "dense", timesteps=T_DENSE)
import shutil
for sf in ["metrics.csv", "model.zip"]:
    src = RESULTS_DIR / f"ppo_dense_{sf}"
    if src.exists(): shutil.copy(src, RESULTS_DIR / f"ppo_dense_level1_{sf}")

In [ ]:
print("Part A — Level 2 world (city_level2.wbt)")
input("Verify Webots has loaded city_level2.wbt, then press enter...")
launch_training("ppo", "dense", timesteps=T_DENSE)
for sf in ["metrics.csv", "model.zip"]:
    src = RESULTS_DIR / f"ppo_dense_{sf}"
    if src.exists(): shutil.copy(src, RESULTS_DIR / f"ppo_dense_level2_{sf}")

In [ ]:
world_labels = {"ppo_dense": "city.wbt (baseline)", "ppo_dense_level1": "city_level1.wbt (medium)", "ppo_dense_level2": "city_level2.wbt (hard)"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key, label in world_labels.items():
    p = RESULTS_DIR / f"{key}_metrics.csv"
    if not p.exists(): continue
    df = pd.read_csv(p)
    for ax, metric in zip(axes, ["mean_reward", "success_rate_%"]):
        ax.plot(df["n_episodes"], df[metric].rolling(5, min_periods=1).mean(), label=label, linewidth=2)
for ax, title in zip(axes, ["Mean Reward", "Success Rate (%)"]):
    ax.set_xlabel("Episodes"); ax.set_title(title); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Part B — Distortion Filters

In [ ]:
import cv2
def apply_fog(img: np.ndarray, severity: float = 0.5) -> np.ndarray:
    fog_color = np.ones_like(img, dtype=np.float32) * 255
    return np.clip(img.astype(np.float32) * (1 - severity) + fog_color * severity, 0, 255).astype(np.uint8)

def apply_rain(img: np.ndarray, n_streaks: int = 200, severity: float = 0.5) -> np.ndarray:
    out = img.copy().astype(np.float32)
    h, w = out.shape[:2]
    rng  = np.random.default_rng(42)
    for _ in range(n_streaks):
        x, y0 = rng.integers(0, w), rng.integers(0, h)
        y1 = min(h, y0 + rng.integers(5, 20))
        alpha = rng.uniform(0.3, 0.7) * severity
        out[y0:y1, max(0, x-1):x+1] = np.clip(out[y0:y1, max(0, x-1):x+1] * (1 - alpha) + 220 * alpha, 0, 255)
    return out.astype(np.uint8)

def apply_low_light(img: np.ndarray, severity: float = 0.6) -> np.ndarray:
    factor = 1.0 - 0.8 * severity
    return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

demo_frame = np.random.randint(80, 200, (64, 64, 3), dtype=np.uint8)
fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
frames = {"Original": demo_frame, "Fog": apply_fog(demo_frame, 0.5), "Rain": apply_rain(demo_frame, 200, 0.6), "Low-light": apply_low_light(demo_frame, 0.6)}
for ax, (title, fr) in zip(axes, frames.items()):
    ax.imshow(fr); ax.set_title(title, fontsize=9); ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "exp3b_filter_preview.png")
plt.show()

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
try:
    from utils import observation as obs_module
    _original_preprocess = obs_module.preprocess_obs
    def make_distorted_preprocess(distort_fn):
        def distorted_preprocess(raw_obs, config):
            raw_obs = dict(raw_obs)
            raw_obs["camera"] = distort_fn(raw_obs["camera"])
            return _original_preprocess(raw_obs, config)
        return distorted_preprocess
    print("✅ Monkey-patching configuration initialized cleanly.")
except ImportError as e:
    print(f"⚠️ Could not import utils.observation offline: {e}")

In [ ]:
DISTORTIONS = ["none", "fog", "rain", "low_light"]
distortion_results = {}
for distortion in DISTORTIONS:
    model_path = RESULTS_DIR / "ppo_dense_model.zip"
    if not model_path.exists(): break
    tmp_cfg = copy.deepcopy(BASE_CFG)
    tmp_cfg["_distortion"] = distortion
    tmp_config_path = RESULTS_DIR / f"_tmp_cfg_{distortion}.yaml"
    with open(tmp_config_path, "w") as f: yaml.dump(tmp_cfg, f)
    
    cmd = [str(WEBOTS_PY), str(PROJECT_ROOT / "evaluate.py"), "--model", str(model_path), "--config", str(tmp_config_path), "--episodes", "10"]
    ret = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_ROOT)
    eval_df = load_eval_csv("ppo_dense_model")
    if not eval_df.empty:
        distortion_results[distortion] = {
            "success_rate_%": 100.0 * (eval_df["collisions"] == 0).mean(),
            "mean_cte":       eval_df["mean_cte"].mean(),
            "mean_steps":     eval_df["total_steps"].mean(),
            "mean_reward":    eval_df["total_reward"].mean(),
        }
print("Evaluation complete.", distortion_results)

In [ ]:
if distortion_results:
    rob_df = pd.DataFrame(distortion_results).T
    display(rob_df)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
    for ax, metric in zip(axes, ["success_rate_%", "mean_cte", "mean_reward"]):
        ax.bar(rob_df.index, rob_df[metric].values, color=["#4CAF50", "#9E9E9E", "#03A9F4", "#FF5722"])
        ax.set_title(metric.replace("_", " "))
    plt.tight_layout()
    plt.show()

## 6. Final Summary Report

In [ ]:
all_cfgs = [("ppo", "dense"), ("ppo", "ttc"), ("ppo", "sparse"),
            ("dqn", "dense"), ("dqn", "ttc"), ("dqn", "sparse")]
final_tbl = summary_table(all_cfgs)
if not final_tbl.empty:
    display(final_tbl.style.background_gradient(cmap="RdYlGn", subset=["success_rate_%"])
                           .background_gradient(cmap="RdYlGn_r", subset=["mean_collisions"])
                           .format("{:.3f}"))

In [ ]:
if not final_tbl.empty:
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.flatten()
    for ax, (agent, reward) in zip(axes, all_cfgs):
        df = load_metrics_csv(agent, reward)
        if df.empty: continue
        ax.plot(df["n_episodes"], df["mean_reward"].rolling(5, min_periods=1).mean())
        ax.set_title(f"{agent.upper()} + {reward}")
    plt.tight_layout()
    plt.show()

In [ ]:
def radar_chart(df, metrics, title="Radar Chart"):
    labels, n = metrics, len(metrics)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist() + [0]
    fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
    for idx, (config_name, row) in enumerate(df.iterrows()):
        vals = []
        for m in metrics:
            col = df[m].dropna()
            v = (float(row[m]) - col.min()) / (col.max() - col.min() + 1e-9)
            if m in ("mean_collisions", "mean_cross_track_error_m"): v = 1.0 - v
            vals.append(v)
        vals += [vals[0]]
        ax.plot(angles, vals, label=config_name)
    ax.set_thetagrids(np.degrees(angles[:-1]), [m.replace("_", "\n") for m in labels], fontsize=8)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)
    plt.show()

if not final_tbl.empty:
    radar_chart(final_tbl, ["success_rate_%", "mean_collisions", "mean_cross_track_error_m", "safety_score"])

In [ ]:
if not final_tbl.empty:
    for config, row in final_tbl.iterrows():
        print(f"\n{config}")
        for col, val in row.items(): print(f"  {col:35s}: {val:.3f}")

## 7. Offline Analysis — Load & Plot Existing CSVs

In [ ]:
print("=== Available Result Logs ===")
for p in sorted(RESULTS_DIR.iterdir()):
    if p.suffix in (".csv", ".zip"):
        print(f"  {p.name:<45} {p.stat().st_size // 1024:>5} KB")

In [ ]:
df_inspect = load_metrics_csv("ppo", "dense")
if not df_inspect.empty: display(df_inspect.describe())

In [ ]:
plot_learning_curves(configs=all_cfgs, metric="success_rate_%", title="Custom Performance Plot", window=3)

---
## Appendix — Notes & Known Issues

### Reward formula correction (dense)
The implementation in `reward.py` uses alignment angle proxies as metric distance counterparts. State explicitly inside your working methodology write-up that `mean_cross_track_error_m` represents image-plane track deviation values, not physical Euclidean distances.

### TTC reward — `d_min` vs. metric distance
The safety factor calculations rely directly on minimum elements extracted dynamically from incoming LiDAR scanner arrays. Calibrate your `d_safe` variable within `config.yaml` to ensure bounds mapping functions resolve reliably.